# Lab 1 · Four agents in one environment

In the lecture we said an agent takes percepts from an environment and chooses actions, and that a rational agent is the one that scores best on the performance measure. This lab turns that into code: you build 4 kinds of agent, drop them into the same environment, and compare their scores.

**Environment: the document inbox.** 24 documents arrive one at a time. The agent must decide what type each one is, or hand it to a person.

| PEAS | In this task |
|---|---|
| **P**erformance | Correct +1, handed to a person 0, wrong −1 |
| **E**nvironment | Thai and English business documents arriving in order |
| **A**ctuators | Choose 1 of 7 actions: 6 document types, or `needs_human` |
| **S**ensors | The text of the current document |

**Time: 70 minutes** · Parts 1–6 are Core and everyone should finish them · Part 7 is Stretch

Look for `# TODO` in the code: that is what you write. Every cell runs even before you change it, but the score will be low. Before you start, save this file as `lab1_inbox_agents.mine.ipynb`.

## 1. Meet the environment (5 min)

`InboxEnvironment` hands documents to the agent one at a time, in order, and scores every action. What the agent sees is a `Percept`, which holds only a `doc_id` and the `text`. The agent never sees the answer key.

In [ ]:
import pandas as pd

from agentlab import get_llm, release_model
from agentlab.inbox import ACTIONS, InboxEnvironment, summarize

env = InboxEnvironment()
print("documents:", len(env.percepts))
print("actions:  ", ACTIONS)
print("rewards:  ", env.rewards)

first = env.percepts[0]
print(f"\n--- {first.doc_id} ---\n{first.text}")

In [ ]:
# The answer key. Agents never see this; only the environment uses it to score.
env.labels[["doc_type", "expected_action", "language", "tricky"]].head(8)

## 2. Simple reflex agent (10 min)

The simplest agent uses **condition–action rules**: if you see this word, do that. It looks only at the current percept, remembers nothing, and does not think ahead.

The agent below has rules for invoices only. **Your job:** add rules for the other 4 types (`receipt`, `purchase_order`, `contract`, `leave_request`), with both Thai and English keywords. Rules are checked in order and the first match wins.

In [ ]:
RULES = [
    # (keyword, action) - checked in order, first match wins
    ("ใบแจ้งหนี้", "invoice"),
    ("invoice", "invoice"),
    # TODO: add rules for receipt, purchase_order, contract and leave_request
]


class SimpleReflexAgent:
    """Condition-action rules on the current percept only."""

    def act(self, percept):
        text = percept.text.lower()
        for keyword, action in RULES:
            if keyword in text:
                return action
        return "other"


runs, agents = {}, {}
agents["simple reflex"] = SimpleReflexAgent()
runs["simple reflex"] = env.run(agents["simple reflex"])
summarize(runs, agents)

See which documents the agent got wrong, then open one or two of them. Ask yourself: which rule caused the mistake, and could another rule fix it?

In [ ]:
def mistakes(name):
    df = runs[name]
    return df[df["outcome"] != "correct"][["expected", "action", "outcome"]]


def show(doc_id):
    print(next(p.text for p in env.percepts if p.doc_id == doc_id))


display(mistakes("simple reflex"))
show("doc_09")

## 3. Model-based reflex agent (10 min)

Documents `doc_14` and `doc_22` are invoices that were **sent a second time** (same number as one that arrived earlier). Passing them on would mean paying twice, so the right action is `needs_human`.

A simple reflex agent cannot do this however many rules you add, because knowing something is a duplicate depends on what was seen before. A **model-based agent** keeps internal state that summarises its percept history.

**Your job:** keep the invoice numbers seen so far in `self.seen`. If a number comes up again, return `needs_human`.

In [ ]:
import re

DOC_NO = re.compile(r"\b[A-Z]{2,3}-\d[\d-]*\b")  # matches INV-2026-0042, TI-55012, ...


class ModelBasedReflexAgent(SimpleReflexAgent):
    """Same rules, plus internal state: the invoice numbers seen so far."""

    def reset(self):
        self.seen = set()

    def act(self, percept):
        action = super().act(percept)
        if action == "invoice":
            match = DOC_NO.search(percept.text)
            # TODO: if this invoice number was seen before, return "needs_human";
            #       otherwise remember it in self.seen
        return action


agents["model-based reflex"] = ModelBasedReflexAgent()
runs["model-based reflex"] = env.run(agents["model-based reflex"])
summarize(runs, agents)

## 4. LLM agent (15 min)

Now the LLM makes the decision. We use structured output to force the answer to be one of the 6 types, so the program can use it directly.

Note that the schema also has a `doc_no` field. This agent does not use it yet, but the next part needs it.

**Your job:** the `SYSTEM` prompt below only names the types. Write a short definition of each, paying attention to documents that *mention* another type (a receipt quoting an invoice number, or a quotation that looks like an invoice).

<small>Running all 24 documents takes about 30–60 seconds on the local model. While tuning the prompt you can use `InboxEnvironment(limit=10)` for shorter rounds.</small>

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class Routing(BaseModel):
    doc_type: Literal["invoice", "receipt", "purchase_order", "contract", "leave_request", "other"]
    doc_no: str | None = Field(description="The document's own number, exactly as printed; null if it has none")


# TODO: replace this with a prompt that defines each type in one line
SYSTEM = "Classify the document as one of: invoice, receipt, purchase_order, contract, leave_request, other."


class LLMAgent:
    """The LLM is the agent function: percept in, action out."""

    def __init__(self, llm):
        self.model = llm.with_structured_output(Routing)
        self.llm_calls = 0

    def reset(self):
        self.llm_calls = 0

    def decide(self, percept) -> Routing:
        self.llm_calls += 1
        return self.model.invoke([("system", SYSTEM), ("user", percept.text)])

    def act(self, percept):
        return self.decide(percept).doc_type


llm = get_llm()
agents["LLM"] = LLMAgent(llm)
runs["LLM"] = env.run(agents["LLM"])
display(summarize(runs, agents))
mistakes("LLM")

## 5. LLM + state (10 min)

The LLM agent reads documents far better than the rules do. But look at its mistakes: it still misses the duplicate invoices, for the reason from Module 0: **an LLM has no memory**. Every request starts from zero.

The fix is what you did in part 3: let *our program* keep the state and use the LLM to read the document. This is how every modern agent is built: the LLM decides, and the code around it looks after the state.

**Your job:** use the `doc_no` the LLM extracted and check it against `self.seen`, as in part 3.

In [ ]:
class StatefulLLMAgent(LLMAgent):
    """LLM for reading, program state for remembering."""

    def reset(self):
        super().reset()
        self.seen = set()

    def act(self, percept):
        decision = self.decide(percept)
        # TODO: if decision.doc_type is "invoice" and decision.doc_no was seen before,
        #       return "needs_human"; otherwise remember decision.doc_no
        return decision.doc_type


agents["LLM + state"] = StatefulLLMAgent(llm)
runs["LLM + state"] = env.run(agents["LLM + state"])
summarize(runs, agents)

## 6. Compare, then extract data from documents (15 min)

### 6.1 Which agent is the most rational?
That depends on the performance measure. First we add one more agent: `CautiousReflexAgent` uses the same rules but acts only when exactly one type matches, and otherwise hands the document to a person.

The table below re-scores the existing results (no need to run the agents again) under three different measures. Edit the numbers in `measures` and see whether the ranking changes. Look also at the `seconds` and `llm_calls` columns in the previous table: those are costs our performance measure does not count yet.

In [ ]:
def matching_types(text):
    text = text.lower()
    return {action for keyword, action in RULES if keyword in text}


class CautiousReflexAgent:
    """Acts only when exactly one type matches; otherwise hands the document to a person."""

    def act(self, percept):
        types = matching_types(percept.text)
        return next(iter(types)) if len(types) == 1 else "needs_human"


agents["cautious reflex"] = CautiousReflexAgent()
runs["cautious reflex"] = env.run(agents["cautious reflex"])

measures = {
    "default (+1, 0, -1)": {"correct": 1, "abstain": 0, "wrong": -1},
    "mistakes are costly (+1, 0, -10)": {"correct": 1, "abstain": 0, "wrong": -10},
    "people are costly (+1, -3, -1)": {"correct": 1, "abstain": -3, "wrong": -1},
}
pd.DataFrame({label: {name: df["outcome"].map(reward).sum() for name, df in runs.items()} for label, reward in measures.items()})

### 6.2 Extract data from the financial documents
Knowing the document type is only the first step. Document intelligence has to pull the data out for later use. We ask the LLM for 4 fields from the 11 financial documents and measure accuracy field by field.

Notice the division of labour: the LLM **copies the date as printed**, and converting it to a standard format is left to code, because a small model often gets the Buddhist-era to Gregorian year conversion wrong.

**Your job:** `to_iso_date` below only knows English month names. Add the Thai months, and convert Buddhist-era years to Gregorian (subtract 543).

In [ ]:
class FinancialDoc(BaseModel):
    doc_no: str = Field(description="The document's own number, exactly as printed")
    issue_date_text: str | None = Field(description="The date the document was issued, copied exactly as printed; null if it has none")
    total: float = Field(description="Final total amount including VAT")
    currency: Literal["THB", "USD", "EUR"]


MONTHS = {m: i for i, m in enumerate(
    ["january", "february", "march", "april", "may", "june", "july", "august", "september", "october", "november", "december"], 1)}
# TODO: add the Thai month names to MONTHS, for example MONTHS["ตุลาคม"] = 10


def to_iso_date(text):
    """'5 October 2026' -> '2026-10-05'. Returns '' when no date is found."""
    match = re.search(r"(\d{1,2})\s+(\S+)\s+(\d{4})", text or "")
    if not match or match.group(2).lower() not in MONTHS:
        return ""
    day, month, year = int(match.group(1)), MONTHS[match.group(2).lower()], int(match.group(3))
    # TODO: a year above 2400 is in the Buddhist era; subtract 543
    return f"{year:04d}-{month:02d}-{day:02d}"


financial = env.labels[env.labels["total"].notna() & (env.labels["expected_action"] != "needs_human")]
texts = {p.doc_id: p.text for p in env.percepts}
extractor = llm.with_structured_output(FinancialDoc)

rows = []
for doc_id in financial.index:
    doc = extractor.invoke([("system", "Extract the fields from this business document."), ("user", texts[doc_id])])
    rows.append({"doc_id": doc_id, "doc_no": doc.doc_no, "issue_date": to_iso_date(doc.issue_date_text),
                 "total": doc.total, "currency": doc.currency, "date_as_printed": doc.issue_date_text})
extracted = pd.DataFrame(rows).set_index("doc_id")
release_model()  # give the model's RAM back to the machine

truth = financial[["doc_no", "issue_date", "total", "currency"]].fillna({"issue_date": ""})
correct = pd.DataFrame({
    "doc_no": extracted["doc_no"].str.strip() == truth["doc_no"],
    "issue_date": extracted["issue_date"] == truth["issue_date"],
    "total": (extracted["total"] - truth["total"]).abs() < 0.005,
    "currency": extracted["currency"] == truth["currency"],
})
print("field accuracy:", correct.mean().round(2).to_dict())
extracted[~correct.all(axis=1)]

## 7. Stretch: a hybrid agent

LLM + state is the most accurate, but it calls the LLM for every document, although many are easy enough for rules. A **hybrid agent** tries the rules first and calls the LLM only when the rules are unsure. This is the System 1 (fast, cheap) and System 2 (slow, careful) idea we return to in Module 3.

`matching_types` from part 6 returns every type that has a matching rule. **Your job:** if exactly one type matches, use it without calling the LLM; otherwise ask the LLM. Then compare the score with the number of `llm_calls`.

If you have time left, also try:
- Turn thinking on with `get_llm(reasoning=True)` and compare score against time
- If you have an API key, try `get_llm("openrouter")` with the same agent

In [ ]:
class HybridAgent(StatefulLLMAgent):
    """Rules when they are sure, the LLM when they are not."""

    def act(self, percept):
        types = matching_types(percept.text)
        # TODO: if exactly one type matches and it is not "invoice", return it without calling the LLM
        return super().act(percept)


agents["hybrid"] = HybridAgent(llm)
runs["hybrid"] = env.run(agents["hybrid"])
summarize(runs, agents)

## Wrap-up and what to hand in

Answer these 4 questions in 2–3 sentences each, and hand them in with a screenshot of the summary table from part 5, as your instructor specifies.

1. What kind of document did the simple reflex agent get wrong, and why can adding rules never fix all of them?
2. Why does an LLM agent that reads documents well still miss the duplicate invoices, and what fixes that?
3. Compare the simple reflex and cautious reflex agents under all three measures. Which is better under each? What does this tell you about designing a performance measure?
4. In part 6.2, which fields did the LLM extract accurately, and which needed help from code?

**From notebook to program:** once you have finished the notebook, open `inbox_app.py` in the same folder. It is this agent written to be run for real, with state saved to a file, results as CSV, and handling for when the LLM fails. The Lab 1 guide shows how to run it.

**Project milestone 1:** your group will use these answers when writing the PEAS description and choosing an agent structure for your own document problem.